# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. All references to record sets, fields, and columns use their `@id` values according to the Croissant schema.

### Dataset Source
The dataset Croissant schema is available at:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}:\n{metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Keywords: {metadata.keywords}")
print(f"License: {metadata.license}")


## 2. Data Overview
Review available record sets and their field `@id`s.

**Note:** All IDs are referenced using the `@id` property from the Croissant schema.

In [ ]:
# List all available record sets and fields by @id
print("Available Record Sets in the dataset (by @id and name):\n")
record_sets = list(metadata.iter_record_sets())
for record_set in record_sets:
    print(f"  @id: {record_set.id}")
    print(f"    Name: {getattr(record_set, 'name', '')}")
    # List fields for the record set
    if hasattr(record_set, 'fields'):
        print(f"    Fields (by @id and name):")
        for field in record_set.fields:
            print(f"      @id: {field.id} | name: {getattr(field, 'name', '')}")
    print("")
if not record_sets:
    print("No record sets found in the metadata.")

## 3. Data Extraction
Load records from a record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# -- Find the record set for protein data --
# Let's programmatically choose the first available record set for demonstration
record_sets = list(metadata.iter_record_sets())
if len(record_sets) == 0:
    raise RuntimeError("No record sets found in this dataset.")
selected_record_set = record_sets[0]
record_set_id = selected_record_set.id

# Show selected record set and its field @ids
print(f"Using Record Set: {record_set_id}")
field_ids = [field.id for field in selected_record_set.fields]
print("Fields (by @id):", field_ids)

# Load the entire record set as a DataFrame
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's explore the data: filter for proteins with high abundance, normalize numeric fields, and aggregate/group by a categorical field.

All field references use the field `@id`. Fields of interest can be chosen from the field list above.

In [ ]:
# Choose a field for numeric analysis. We'll pick one by searching for likely numeric columns.
import numpy as np

# Try find a field that may represent protein 'abundance', 'coverage', or similar
numeric_candidates = [
    col for col in df.columns if any(x in col.lower() for x in ['abundance', 'coverage', 'count', 'mw', 'weight', 'score', 'peptide', 'site'])
    and np.issubdtype(df[col].dtype, np.number)
]
if len(numeric_candidates) == 0:
    # Fallback on any numeric field
    numeric_candidates = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
if len(numeric_candidates) == 0:
    raise RuntimeError("No numeric fields found in the record set DataFrame.")
numeric_field = numeric_candidates[0]
print(f"Selected numeric field (by @id): {numeric_field}")

# Filter: keep those above the 75th percentile
threshold = df[numeric_field].quantile(0.75)
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records where {numeric_field} > {threshold:.2f} (75th percentile): {len(filtered_df)} rows")

# Normalize
filtered_df[numeric_field + "_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

# Try grouping by a categorical field (e.g., 'description', 'accession', etc, pick by string columns not used as numeric_field)
non_num_cols = [c for c in df.columns if df[c].dtype == object and c != numeric_field]
group_field = None
for col in non_num_cols:
    # Exclude accession-likes as they are unique per row
    if not any(x in col.lower() for x in ['accession', 'id', 'uid', 'uniprot']):
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).to_frame()
    print(f"\nGrouped data by {group_field} (showing mean of {numeric_field}):")
    print(grouped_df.head())
else:
    print("No categorical text field suitable for grouping found.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field (by `@id`) and its normalized values.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(df[numeric_field].dropna(), bins=30, color='deepskyblue', alpha=0.7, label=numeric_field)
plt.axvline(threshold, color='orange', linestyle='dashed', linewidth=2, label=f'Threshold={threshold:.2f}')
plt.title(f"Distribution of {numeric_field} (by @id)")
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.legend()
plt.show()

if numeric_field + "_normalized" in filtered_df.columns:
    plt.figure(figsize=(8, 4))
    plt.hist(filtered_df[numeric_field + "_normalized"].dropna(), bins=30, color='tomato', alpha=0.7)
    plt.title(f"Distribution of Normalized {numeric_field} (filtered)")
    plt.xlabel(f"{numeric_field}_normalized")
    plt.ylabel('Frequency')
    plt.show()

## 6. Conclusion
In this notebook, we loaded protein abundance and related data from a mass spectrometry Croissant dataset using the `mlcroissant` library. Fields and record sets were referenced by their Croissant `@id`, and initial exploratory analysis plus visualizations were performed. This approach can be adapted to your own datasets by changing the Croissant URL and referencing the appropriate `@id` values in each step.

- Dataset Title: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
- Croissant schema: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

Explore further by modifying filters, normalizations, and visualizations according to your research questions!